## Import libraries and metrics for TON_IoT IDS

In [ ]:
# Here this Python 3 environment comes with many helpful analytics libraries installed


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import StandardScaler
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier




import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/iot-network/TON_IoT.csv


## Load TON_IoT dataset and show label distribution

In [11]:
df = pd.read_csv("/kaggle/input/iot-network/TON_IoT.csv")
print("Columns:", df.columns)
print("\nLabel counts:")
print(df['label'].value_counts())

Columns: Index(['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service',
       'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes',
       'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query',
       'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA',
       'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed',
       'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth',
       'http_method', 'http_uri', 'http_version', 'http_request_body_len',
       'http_response_body_len', 'http_status_code', 'http_user_agent',
       'http_orig_mime_types', 'http_resp_mime_types', 'weird_name',
       'weird_addl', 'weird_notice', 'label', 'type'],
      dtype='object')

Label counts:
label
1    161043
0     50000
Name: count, dtype: int64


## Define preprocessing pipeline (label detection, feature extraction, scaling)

In [12]:
POSSIBLE_LABEL_COLS = ["label", "class", "Label", "Class", "attack", "target"]

def find_label_column(df: pd.DataFrame) -> str:
    for col in POSSIBLE_LABEL_COLS:
        if col in df.columns:
            return col
    raise ValueError(f"Could not find any label column in {POSSIBLE_LABEL_COLS}")


def split_features_labels(df: pd.DataFrame, label_col: str):
    y_raw = df[label_col]

    # If labels are text like "Normal", "Attack"
    if y_raw.dtype == "object":
        y = (y_raw.astype(str).str.lower() != "normal").astype(int)
    else:
        # numeric: 0 = normal, non-zero = attack
        y = (y_raw != 0).astype(int)

    X = df.drop(columns=[label_col])
    X = X.select_dtypes(include=[np.number]).fillna(0)
    return X, y


def scale_features(X: pd.DataFrame):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler


def prepare_data(df):
    label_col = find_label_column(df)
    X, y = split_features_labels(df, label_col)
    X_scaled, scaler = scale_features(X)
    feature_names = list(X.columns)
    return X_scaled, y.to_numpy(), scaler, feature_names, label_col


X_scaled, y, scaler, feature_names, label_col = prepare_data(df)
print("Data prepared!")
print("Number of samples:", X_scaled.shape[0])
print("Number of features:", X_scaled.shape[1])
print("Label column:", label_col)


Data prepared!
Number of samples: 211043
Number of features: 16
Label column: label


## Define training function for RandomForest classifier

In [ ]:
def train_random_forest(X, y, random_state: int = 42):
    clf = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        random_state=random_state,
        n_jobs=-1,
        class_weight="balanced",
    )
    clf.fit(X, y)
    return clf  
                 #works well on imbalanced data
                #Handles non-linear patterns


## Define training function for Logistic Regression classifier

In [14]:

def train_logistic_regression(X, y, random_state: int = 42):
    clf = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=random_state,
        n_jobs=-1
    )
    clf.fit(X, y)
    return clf


## Define training function for SVC (RBF kernel) classifier

In [15]:

def train_svc(X, y, random_state: int = 42):
    clf = SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=random_state
    )
    clf.fit(X, y)
    return clf


## Define training function for XGBoost classifier

In [ ]:

def train_xgboost(X, y, random_state: int = 42):
    clf = XGBClassifier(
        n_estimators=300,  # Keep 300 boosting rounds
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=random_state,
        tree_method="hist",
        eval_metric="logloss"
    )
    clf.fit(X, y)
    return clf
                    # XGBoost is powerful
                    # Excellent with tabular data
                    # Handles complex patterns
                    # good for large amout of data

## Define training function for Gradient Boosting classifier

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

def train_gradient_boosting(X, y, random_state: int = 42):
    clf = GradientBoostingClassifier(
        random_state=random_state
    )
    clf.fit(X, y)
    return clf
                #also faster , good for small amout of data


## Define evaluation function and create train/test split

In [ ]:

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    auc = None
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print("=== Evaluation on Test Set ===")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1-score :", f1)
    if auc is not None:
        print("ROC-AUC  :", auc)
    print()
    print("=== Classification Report ===")
    print(classification_report(y_test, y_pred, digits=4))

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": auc,
    }
 
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)


## Train and evaluate RandomForest on TON_IoT

In [19]:
rf = train_random_forest(X_train, y_train)

_ = evaluate_model(rf, X_test, y_test)

=== Evaluation on Test Set ===
Accuracy : 0.9987443436233978
Precision: 0.9991617510090034
Recall   : 0.9991927722065261
F1-score : 0.999177261366988
ROC-AUC  : 0.9999486727312241

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.9974    0.9973    0.9973     10000
           1     0.9992    0.9992    0.9992     32209

    accuracy                         0.9987     42209
   macro avg     0.9983    0.9982    0.9983     42209
weighted avg     0.9987    0.9987    0.9987     42209



## Train and evaluate Logistic Regression on TON_IoT

In [20]:
lr =  train_logistic_regression(X_train, y_train)
_ = evaluate_model(lr, X_test, y_test)

=== Evaluation on Test Set ===
Accuracy : 0.8525906797128574
Precision: 0.885369398226414
Recall   : 0.9268216957993107
F1-score : 0.9056214543579164
ROC-AUC  : 0.8790309416622684

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.7224    0.6135    0.6635     10000
           1     0.8854    0.9268    0.9056     32209

    accuracy                         0.8526     42209
   macro avg     0.8039    0.7702    0.7846     42209
weighted avg     0.8468    0.8526    0.8483     42209



## Train and evaluate XGBoost on TON_IoT

In [21]:
xg = train_xgboost(X_train, y_train)
_ = evaluate_model(xg, X_test, y_test)

=== Evaluation on Test Set ===
Accuracy : 0.9982705110284537
Precision: 0.9989442305303688
Recall   : 0.9987891583097892
F1-score : 0.9988666884014097
ROC-AUC  : 0.9999648126300104

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.9961    0.9966    0.9964     10000
           1     0.9989    0.9988    0.9989     32209

    accuracy                         0.9983     42209
   macro avg     0.9975    0.9977    0.9976     42209
weighted avg     0.9983    0.9983    0.9983     42209



## Train and evaluate Gradient Boosting on TON_IoT

In [22]:
gb = train_gradient_boosting(X_train, y_train)
_ = evaluate_model(gb, X_test, y_test)

=== Evaluation on Test Set ===
Accuracy : 0.994029709303703
Precision: 0.9941243776478956
Recall   : 0.9980750721847931
F1-score : 0.9960958076410622
ROC-AUC  : 0.9991835915427366

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.9937    0.9810    0.9873     10000
           1     0.9941    0.9981    0.9961     32209

    accuracy                         0.9940     42209
   macro avg     0.9939    0.9895    0.9917     42209
weighted avg     0.9940    0.9940    0.9940     42209



## Save trained RandomForest model and preprocessing info as rf_model

In [ ]:
def save_model(model, scaler, feature_names, label_col):
    to_save = {
        "model": model,
        "scaler": scaler,
        "feature_names": feature_names,
        "label_col": label_col,
    }
    joblib.dump(to_save,filename='rf_model')
    print(f"Model saved to /kaggle/working/")

save_model(rf, scaler, feature_names, label_col)         #helps to deployment

Model saved to /kaggle/working/


## Inspect y_test (test labels) 

In [24]:
y_test

array([1, 1, 0, ..., 1, 1, 1])

## Inspect X_test (test feature matrix)

In [25]:
X_test

array([[-0.28675894, -0.33509494, -0.0136461 , ..., -0.00707727,
        -0.0047561 , -0.03674628],
       [ 0.55862405, -0.33509494, -0.01365045, ..., -0.00707727,
        -0.0047561 , -0.03674628],
       [-0.06083316, -0.33774418, -0.01365065, ..., -0.00707727,
        -0.0047561 , -0.03674628],
       ...,
       [-2.0015056 , -0.34265018, -0.01365065, ..., -0.00707727,
        -0.0047561 , -0.03674628],
       [ 0.90440134, -0.33509494, -0.01364982, ..., -0.00707727,
        -0.0047561 , -0.03674628],
       [ 0.56530549,  0.44986518, -0.01365054, ..., -0.00707727,
        -0.0047561 , -0.03674628]])

## Load saved rf_model and run sample-wise real-time predictions

In [26]:
MODEL_PATH = "/kaggle/working/rf_model"


def load_ids_model(path: str = MODEL_PATH):
    saved = joblib.load(path)
    model = saved["model"]
    scaler = saved["scaler"]
    feature_names = saved["feature_names"]
    label_col = saved["label_col"]
    return model, scaler, feature_names, label_col


def preprocess_rows(df: pd.DataFrame, feature_names, scaler):
    for col in ["label", "class", "Label", "Class", "attack", "target"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    df = df.select_dtypes(include=[np.number]).fillna(0)

    for col in feature_names:
        if col not in df.columns:
            df[col] = 0.0

    df = df[feature_names]
    X_scaled = scaler.transform(df.values)
    return X_scaled


print(f"Loading model from {MODEL_PATH}...")
model, scaler, feature_names, label_col = load_ids_model(MODEL_PATH)

df_sample = df.sample(frac=0.001, random_state=69)  

has_label = label_col in df_sample.columns

X_scaled = preprocess_rows(df_sample, feature_names, scaler)


for i, x in enumerate(X_scaled):
    x_2d = x.reshape(1, -1)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(x_2d)[0, 1]
        pred = 1 if proba >= 0.5 else 0
    else:
        pred = model.predict(x_2d)[0]
        proba = None

    label_str = "ATTACK" if pred == 1 else "NORMAL"

    msg = f"Sample {i:05d}: {label_str}"
    if proba is not None:
        msg += f" (attack probability: {proba:.3f})"

    if has_label:
        true_label = df.iloc[i][label_col]
        msg += f" | true: {label_str}"

    print(msg)

print("\nFinished processing all samples.")





Loading model from /kaggle/working/rf_model...
Sample 00000: ATTACK (attack probability: 1.000) | true: ATTACK
Sample 00001: NORMAL (attack probability: 0.000) | true: NORMAL
Sample 00002: NORMAL (attack probability: 0.000) | true: NORMAL
Sample 00003: NORMAL (attack probability: 0.000) | true: NORMAL
Sample 00004: ATTACK (attack probability: 0.999) | true: ATTACK


/usr/local/lib/python3.11/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Sample 00005: ATTACK (attack probability: 1.000) | true: ATTACK
Sample 00006: NORMAL (attack probability: 0.000) | true: NORMAL
Sample 00007: ATTACK (attack probability: 1.000) | true: ATTACK
Sample 00008: ATTACK (attack probability: 1.000) | true: ATTACK
Sample 00009: ATTACK (attack probability: 1.000) | true: ATTACK
Sample 00010: NORMAL (attack probability: 0.000) | true: NORMAL
Sample 00011: ATTACK (attack probability: 1.000) | true: ATTACK
Sample 00012: NORMAL (attack probability: 0.014) | true: NORMAL
Sample 00013: ATTACK (attack probability: 1.000) | true: ATTACK
Sample 00014: ATTACK (attack probability: 0.999) | true: ATTACK
Sample 00015: ATTACK (attack probability: 1.000) | true: ATTACK
Sample 00016: ATTACK (attack probability: 0.999) | true: ATTACK
Sample 00017: ATTACK (attack probability: 1.000) | true: ATTACK
Sample 00018: ATTACK (attack probability: 1.000) | true: ATTACK
Sample 00019: ATTACK (attack probability: 0.999) | true: ATTACK
Sample 00020: NORMAL (attack probability

## Define predict_one() helper for single-row IDS prediction

In [27]:
def predict_one(df, feature_names, scaler, model, index, label_col=None):
    # Extract the row
    row = df.iloc[index]

    # Preprocess it (make it 2D)
    x = preprocess_rows(df.iloc[[index]], feature_names, scaler)[0].reshape(1, -1)

    # Predict
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(x)[0, 1]
        pred = 1 if proba >= 0.5 else 0
    else:
        pred = model.predict(x)[0]
        proba = None

    # Convert to text
    label_str = "ATTACK" if pred == 1 else "NORMAL"

    # Build message
    msg = f"Row {index}: {label_str}"
    if proba is not None:
        msg += f" (attack probability: {proba:.3f})"

    # If dataset has true label
    if label_col is not None:
        true_value = row[label_col]
        if isinstance(true_value, str):
            true_is_attack = (true_value.lower() != "normal")
        else:
            true_is_attack = (true_value != 0)

        true_label_str = "ATTACK" if true_is_attack else "NORMAL"
        msg += f" | true: {true_label_str}"

    return msg
